# Ungraded lab: Distributed Training with Titanic and Random Forest

In this lab you will perform distributed training using **Ray** with a **RandomForestClassifier** on the **Titanic** dataset. You will:
1. Perform training with a single worker.
2. Understand multi-worker setup with Ray.
3. Perform multi-worker distributed training.

## Setup

In [ ]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
if '.' not in sys.path:
    sys.path.insert(0, '.')

## Dataset and Model Definition

Create a `titanic.py` file with data loading and model setup.

In [ ]:
%%writefile titanic.py
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def titanic_dataset():
    df = sns.load_dataset('titanic')
    df.drop(['class','who','adult_male','deck','embark_town','alive','alone'], axis=1, inplace=True)
    df['age'].fillna(df['age'].median(), inplace=True)
    df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
    df['sex'] = df['sex'].map({'male':1,'female':0})
    df['embarked'] = LabelEncoder().fit_transform(df['embarked'])
    X = df.drop('survived', axis=1)
    y = df['survived']
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

def build_and_compile_rf_model():
    model = RandomForestClassifier(n_estimators=100, random_state=42, criterion='entropy')
    return model

In [ ]:
!ls *.py

Import the `titanic` module and try training the model for a single worker to verify everything works.

In [ ]:
import titanic
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = titanic.titanic_dataset()
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

single_worker_model = titanic.build_and_compile_rf_model()
single_worker_model.fit(X_train_s, y_train)
acc = accuracy_score(y_test, single_worker_model.predict(X_test_s))
print(f"Single worker accuracy: {acc:.4f}")

Everything is working as expected! Now you will see how multiple workers can be used as a distributed strategy.

## Multi-worker Configuration

For distributed training with Ray, the `ray.init()` call starts a local cluster. In a real multi-machine setup you would pass `address='ray://<head-node>:10001'`.

Ray's core concepts:
- **Workers**: independent processes that execute remote tasks
- **Object store**: shared memory accessible to all workers via `ray.put()` / `ray.get()`
- **Remote tasks**: functions decorated with `@ray.remote` that run in parallel

In [ ]:
import ray, time

if ray.is_initialized():
    ray.shutdown()

ray.init()
print(ray.cluster_resources())

## Choose the Right Strategy

Ray uses **task parallelism** — each worker trains an independent model. This is equivalent to data-parallel distributed training. All workers share data via the object store.

### Store Data in the Object Store

In [ ]:
X_train_ref = ray.put(X_train_s)
X_test_ref  = ray.put(X_test_s)
y_train_ref = ray.put(y_train)
y_test_ref  = ray.put(y_test)
print(X_train_ref)

## Create Training Script

Create `main.py` that each worker will run:

In [ ]:
%%writefile main.py
import ray
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import titanic

PER_WORKER_N_ESTIMATORS = 50
NUM_WORKERS = 2

if ray.is_initialized():
    ray.shutdown()
ray.init()

X_train, X_test, y_train, y_test = titanic.titanic_dataset()
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

X_train_ref = ray.put(X_train_s)
X_test_ref  = ray.put(X_test_s)
y_train_ref = ray.put(y_train)
y_test_ref  = ray.put(y_test)

@ray.remote
def train_worker(X_train, X_test, y_train, y_test, n_estimators, worker_id):
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=worker_id)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"Worker {worker_id}: n_estimators={n_estimators}, accuracy={acc:.4f}")
    return worker_id, acc

global_n_estimators = PER_WORKER_N_ESTIMATORS * NUM_WORKERS
results_ref = [
    train_worker.remote(X_train_ref, X_test_ref, y_train_ref, y_test_ref,
                        PER_WORKER_N_ESTIMATORS, i)
    for i in range(NUM_WORKERS)
]
results = ray.get(results_ref)
print(f"Results: {results}")
ray.shutdown()

In [ ]:
!ls *.py

## Train the Model

### Launch Workers

Now launch worker processes. Worker 0 is the chief worker.

In [ ]:
%%bash --bg
python main.py &> job_0.log

In [ ]:
import time
time.sleep(15)

In [ ]:
%%bash
cat job_0.log

## Analyze Results

Distributed training completes with both workers contributing to the overall model ensemble.

In [ ]:
@ray.remote
def train_worker(X_train, X_test, y_train, y_test, n_estimators, worker_id):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=worker_id)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"Worker {worker_id}: n_estimators={n_estimators}, accuracy={acc:.4f}")
    return worker_id, acc

results_ref = [
    train_worker.remote(X_train_ref, X_test_ref, y_train_ref, y_test_ref, 50, i)
    for i in range(2)
]
results = ray.get(results_ref)
for worker_id, acc in results:
    print(f"Worker {worker_id} accuracy: {acc:.4f}")

ray.shutdown()

Running multiple workers on a single machine adds overhead. The real benefit is seen on multi-machine clusters where each worker runs on a different node.

**Congratulations on finishing this ungraded lab!**